In [2]:
import numpy as np

K = 5
M = 3

alpha = 0.00001
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

N_0 = 1e-9
P_t = 10

h_k = np.random.rand(M, K)
P = np.random.rand(K, M)
c = np.random.rand(K)
u = np.random.rand(K)
gamma_k = np.random.rand(K)

m_P, v_P = np.zeros_like(P), np.zeros_like(P)
m_c, v_c = np.zeros_like(c), np.zeros_like(c)

Rth_s = np.random.rand(K)
w_secrecy = 10

num_iterations = 1000

def secrecy_rate(P, h_k, gamma_k, N_0):
    K, M = P.shape
    R_s = np.zeros(K)
    for k in range(K):
        h_k_user = h_k[:, k]
        P_k = P[k, :]
        interference = np.sum(np.abs(np.dot(h_k.T, P_k)) ** 2) - np.abs(np.dot(h_k_user.T, P_k))**2
        R_s[k] = np.log2(1 + np.abs(np.dot(h_k_user, P_k))**2 / (N_0 + interference))
    return R_s

def transmit_power_constraint(P, P_t):
    power = np.trace(np.dot(P, P.T))
    return power <= P_t

def non_negativity_check(c):
    return np.all(c >= 0)

penalty_factor = 100

for t in range(1, num_iterations+1):
    grad_P = np.zeros_like(P)
    for k in range(K):
        grad_R_s_k = (1 / np.log(2)) * (2 * np.real(np.dot(h_k[:, k].T, P[k]))) / (
            (1 + gamma_k[k]) * (np.sum(np.abs(np.dot(h_k.T, P[k])) ** 2) + N_0))
        grad_P[k] = u[k] * grad_R_s_k + w_secrecy * grad_R_s_k * 2

    grad_c = u + w_secrecy * grad_R_s_k

    m_P = beta1 * m_P + (1 - beta1) * grad_P
    v_P = beta2 * v_P + (1 - beta2) * grad_P ** 2

    m_P_hat = m_P / (1 - beta1 ** t)
    v_P_hat = v_P / (1 - beta2 ** t)

    P = P - alpha * m_P_hat / (np.sqrt(v_P_hat) + epsilon)

    m_c = beta1 * m_c + (1 - beta1) * grad_c
    v_c = beta2 * v_c + (1 - beta2) * grad_c ** 2

    m_c_hat = m_c / (1 - beta1 ** t)
    v_c_hat = v_c / (1 - beta2 ** t)

    c = c - alpha * m_c_hat / (np.sqrt(v_c_hat) + epsilon)

    R_s = secrecy_rate(P, h_k, gamma_k, N_0)
    secrecy_penalty = np.sum(np.maximum(0, Rth_s - R_s))
    total_loss = secrecy_penalty + penalty_factor * secrecy_penalty

    if t % 100 == 0:
        print(f"Secrecy Rate Penalty: {secrecy_penalty}")

    secrecy_constraint_satisfied = np.all(R_s >= Rth_s)
    power_constraint_satisfied = transmit_power_constraint(P, P_t)
    non_negativity_satisfied = non_negativity_check(c)

    if t % 100 == 0:
        print(f"Transmit Power Constraint Satisfied: {power_constraint_satisfied}")
        print(f"Non-Negativity of Rate Allocation Satisfied: {non_negativity_satisfied}")

print("\nFinal optimized precoder matrix (P):")
print(P)

print("\nFinal optimized rate allocation vector (c):")
print(c)



Secrecy Rate Penalty: 0.8784157310195119
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8778455732502219
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8772729955977252
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8766980381740568
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8761207424513926
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8755411511975932
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8749593084176874
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: True
Secrecy Rate Penalty: 0.8743752592955545
Transmit Power Constraint Sa